In [1]:
import json, os, pickle
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd

import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
import qiskit.circuit.random
import torch, random
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn

import numpy as np
import json, os, pickle
from tqdm import tqdm
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from qiskit import QuantumCircuit

import sys
sys.path.append('../../tutorials/')
from mlp import encode_data, encode_data_v2_ecr

In [2]:
def check_f(f, f_ext, step_indices):
    return f.endswith(f_ext) and any([f"step_%02d"%step_index in f for step_index in step_indices])

def load_circuits(data_dir, step_indices, f_ext='.pk'):
    Js = []
    circuits = []
    data_paths = []
    data_files = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if check_f(f, f_ext, step_indices)])
    for data_file in tqdm(data_files, leave=True):
        for entry in pickle.load(open(data_file, 'rb')):
            Js.append(entry['J'])
            circuits.append(entry['circuit'])
        data_paths.append(data_file)
    return data_paths, circuits, Js

In [3]:
data_paths, circuits, Js = load_circuits('../../tutorials/data/ising_zne_hardware/100q_brisbane/', list(range(1, 11)))

100%|█████████████████████████████████████████████████████████████████████████| 500/500 [00:42<00:00, 11.71it/s]


In [4]:
for step_index in [1]:
    with open('../../tutorials/zne_mitigated/twirl_100q_brisbane/step%02d.json'%step_index, 'r') as file:
        loaded = json.load(file)
    noise_factor_1 = np.array(loaded['noise_factor_1'])
    noise_factor_3 = np.array(loaded['noise_factor_3'])

for step_index in tqdm([2, 3, 4, 5, 6, 7, 8, 9, 10]):
    with open('../../tutorials/zne_mitigated/twirl_100q_brisbane/step%02d.json'%step_index, 'r') as file:
        loaded = json.load(file)
    noise_factor_1 = np.concatenate([noise_factor_1, loaded['noise_factor_1']])
    noise_factor_3 = np.concatenate([noise_factor_3, loaded['noise_factor_3']])

noise_factor_1_tw_avg = noise_factor_1.reshape(noise_factor_1.shape[0], 5, 5).mean(axis=-1)
noise_factor_3_tw_avg = noise_factor_3.reshape(noise_factor_3.shape[0], 5, 5).mean(axis=-1)

slope = (noise_factor_3_tw_avg - noise_factor_1_tw_avg) / 2
zne_mitigated_vals = (noise_factor_1_tw_avg - slope).tolist()
noisy_vals = noise_factor_1_tw_avg.tolist()
len(zne_mitigated_vals)

100%|███████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 1171.70it/s]


500

In [5]:
print(len(circuits), len(zne_mitigated_vals), len(noisy_vals), len(data_paths))

500 500 500 500


In [6]:
import pandas as pd
import json

# Example lists (replace with yours)
# paths  = ["path/to/circ1.qasm", "path/to/circ2.qasm", ...]
# noisy  = [[0.1, -0.2, 0.3, 0.4, -0.5], [...], ...]
# ideal  = [[0.12, -0.18, 0.29, 0.39, -0.49], [...], ...]

df = pd.DataFrame({
    "circuit_path": data_paths,
    "noisy_z_json": [json.dumps(vals) for vals in noisy_vals],
    "target_y_json": [json.dumps(vals) for vals in zne_mitigated_vals],
})

df.to_csv("labels_train.csv", index=False)
print("Wrote labels_train.csv with", len(df), "rows")
print(df.head())

Wrote labels_train.csv with 500 rows
                                        circuit_path  \
0  ../../tutorials/data/ising_zne_hardware/100q_b...   
1  ../../tutorials/data/ising_zne_hardware/100q_b...   
2  ../../tutorials/data/ising_zne_hardware/100q_b...   
3  ../../tutorials/data/ising_zne_hardware/100q_b...   
4  ../../tutorials/data/ising_zne_hardware/100q_b...   

                                        noisy_z_json  \
0  [0.02324, -0.019200000000000002, -0.0258400000...   
1  [-0.44044, -0.4678, -0.4474, -0.45355999999999...   
2  [-0.44487999999999994, -0.4578399999999999, -0...   
3  [-0.44159999999999994, -0.43567999999999996, -...   
4  [-0.42835999999999996, -0.42952, -0.42076, -0....   

                                       target_y_json  
0  [0.014979999999999999, -0.03276, -0.0136600000...  
1  [-0.4677, -0.50136, -0.46118000000000003, -0.4...  
2  [-0.46919999999999995, -0.4794599999999999, -0...  
3  [-0.4755599999999999, -0.4480599999999999, -0....  
4  [-0.4806199